# Vietnamese Embedding Models — Document Retrieval Evaluation
## Dataset: [YuITC/Vietnamese-Legal-Documents](https://huggingface.co/datasets/YuITC/Vietnamese-Legal-Documents)

This notebook evaluates Vietnamese embedding models on a real Vietnamese legal document retrieval task.

### Dataset Structure
| Column | Description |
|---|---|
| `question` | Natural language query (Vietnamese legal question) |
| `context_list` | List of relevant document passages (ground-truth answers) |
| `qid` | Unique query ID |
| `cid` | List of corpus document IDs that are relevant |

### How Retrieval Evaluation Works
1. **Indexing**: All unique passages from `context_list` across the dataset are pooled into one corpus. It will be embedded by different models for evaluating.
2. **Query**: Each `question` is a query which will be embedded using the same indexing model.
3. **Retrieval**: Each model encodes queries, then semantic search by cosine similarity. Usually only take best-K results.
4. **Metrics**: Computed by comparing ranked results against `cid` of the ground truth passage.

### Retrieval Metrics
| Metric | What it measures |
|---|---|
| **nDCG@k** | Ranking quality of top-k results — the primary IR metric |
| **MAP@k** | Mean Average Precision — rewards early, complete retrieval |
| **MRR@k** | Mean Reciprocal Rank — how high is the first relevant doc? |
| **Recall@k** | Fraction of all relevant docs found in top-k |
| **Hit@k** | Fraction of queries where any relevant doc appears in top-k |

## 1. System Runtime Environments

In [1]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

PyTorch version : 2.11.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3060 Laptop GPU


## 2. Load the Dataset

In this evaluation, we use `test` split for faster processing and less burden for hardware.

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np

print("Loading YuITC/Vietnamese-Legal-Documents ...")
raw = load_dataset("YuITC/Vietnamese-Legal-Documents")
print(raw)

# Use the test split for evaluation; fall back to train if test is absent
split = "test" if "test" in raw else list(raw.keys())[0]
df = raw[split].to_pandas()

print(f"\nUsing split : '{split}'")
print(f"Rows        : {len(df):,}")
print(f"Columns     : {df.columns.tolist()}")
df.head(3)

/home/ntdatwows2003/Documents/vi_embed_eval/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading YuITC/Vietnamese-Legal-Documents ...


DatasetDict({
    train: Dataset({
        features: ['question', 'context_list', 'qid', 'cid'],
        num_rows: 89261
    })
    test: Dataset({
        features: ['question', 'context_list', 'qid', 'cid'],
        num_rows: 29746
    })
})

Using split : 'test'
Rows        : 29,746
Columns     : ['question', 'context_list', 'qid', 'cid']


,question,context_list,qid,cid
0,Phó Tổng Giám đốc Ngân hàng Chính sách xã hội ...,[Áp dụng chế độ tiền lương và phụ cấp quy định...,70867,[140864]
1,Ai có thẩm quyền quyết định thành lập Hội đồng...,[Thành lập Hội đồng\n1. Bộ trưởng Bộ Y tế ra q...,813,[62339]
2,Thời hiệu xử phạt đối với nhà xuất bản thực hi...,[Điều 5. Thời hiệu xử phạt vi phạm hành chính\...,40392,[63171]


## 3. Build Relevance Benchmark Table

Remove `context_list` from the dataset for better performance and faster processing. All the unique passages from this field is already encoded inside vector database for retrieval.

Only `cid` is needed to compare model's prediction against ground truth.

In [3]:
# Modify dataframe to drop context_list column
df.drop(columns="context_list", inplace=True)

print(f"Total queries                : {len(df):,}")

Total queries                : 29,746


## 4. Define Models to Evaluate

In [4]:
MODELS = [
    {
        "name": "AITeamVN/Vietnamese_Embedding",
        "label": "AITeamVN",
        # BGE-M3 fine-tuned on 300K Vietnamese triplets; 2048-token context
    },
    # {
    #     "name": "dangvantuan/vietnamese-document-embedding",
    #     "label": "VN-DocEmbed",
    #     "trust_remote_code": True,
    #     # Best for long passages; 8096-token context window
    # },
    # {
    #     "name": "Alibaba-NLP/gte-multilingual-base",
    #     "label": "Alibaba-GTE",
    #     "trust_remote_code": True,
    #     # Multilingual high-performance model in General Text Embedding
    # },
    # {
    #     "name": "BAAI/bge-m3",
    #     "label": "BGE-M3 (multilingual)",
    #     # General multilingual baseline
    # },
]

print(f"Models to evaluate: {len(MODELS)}")
for m in MODELS:
    print(f"  • {m['label']}")

Models to evaluate: 1
  • AITeamVN


## 5. Initialize Vector Database

In [5]:
import chromadb

client = chromadb.PersistentClient(path="./database")

## 6. Metric Functions

In [6]:
def dcg_at_k(ranked_rel, k):
    return sum(r / np.log2(i + 2) for i, r in enumerate(ranked_rel[:k]))

def ndcg_at_k(ranked_rel, k, n_relevant):
    ideal = [1] * min(n_relevant, k) + [0] * max(0, k - n_relevant)
    idcg  = dcg_at_k(ideal, k)
    return dcg_at_k(ranked_rel, k) / idcg if idcg > 0 else 0.0

def map_at_k(ranked_rel, k, n_relevant):
    if n_relevant == 0:
        return 0.0
    hits = ap = 0
    for i, r in enumerate(ranked_rel[:k]):
        if r:
            hits += 1
            ap   += hits / (i + 1)
    return ap / min(n_relevant, k)

def mrr_at_k(ranked_rel, k):
    for i, r in enumerate(ranked_rel[:k]):
        if r:
            return 1.0 / (i + 1)
    return 0.0

def recall_at_k(ranked_rel, k, n_relevant):
    return sum(ranked_rel[:k]) / n_relevant if n_relevant > 0 else 0.0

def precision_at_k(ranked_rel, k):
    return sum(ranked_rel[:k]) / k if k > 0 else 0.0

def all_metrics_at_k(ranked_rel, k, n_relevant):
    return {
        f"nDCG@{k}"      : ndcg_at_k(ranked_rel, k, n_relevant),
        f"MAP@{k}"       : map_at_k(ranked_rel, k, n_relevant),
        f"MRR@{k}"       : mrr_at_k(ranked_rel, k),
        f"Recall@{k}"    : recall_at_k(ranked_rel, k, n_relevant),
        f"Precision@{k}" : precision_at_k(ranked_rel, k),
    }

print("Metric functions ready.")

Metric functions ready.


## 7. Run Retrieval Evaluation

In [12]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from tqdm import tqdm
import time

try:
    from pyvi.ViTokenizer import tokenize as vi_tokenize
    print("pyvi loaded — Vietnamese word segmentation available.")
except ImportError:
    vi_tokenize = None
    print("⚠️  pyvi not found — models requiring it will be skipped.")

K_VALUES   = [1, 5, 10]  # evaluate at these cutoffs
BATCH_SIZE = 64               # reduce if you hit OOM
CHUNK      = 256              # queries processed per similarity batch

all_results    = []   # one dict of avg metrics per model

for model in MODELS:
    print(f"\n{'='*65}")
    print(f"  {model['label']}  ({model['name']})")
    print(f"{'='*65}")

    if model.get("needs_pyvi") and vi_tokenize is None:
        print("  ⚠️  Skipping: pyvi is required but not installed.")
        continue
    
    collection = client.get_collection(model["label"])

    try:
        instances_metrics = []
        for r_idx, data in tqdm(df.iterrows(), total=len(df), desc="Queries processed"):
            # Number of relevant documents
            n_relevant = len(data['cid'])
            
            retrieve_results = collection.query(
                query_texts=[data['question']],
                n_results=10,
            )
            
            # Ranked relevant score from retrival results
            ranked_rel = list(map(lambda x: int(int(x) in data['cid']) , retrieve_results["ids"][0]))
            # print(ranked_rel)
            
            # All recorded metrics for this particular instance
            all_metrics = {}
            for k in K_VALUES:
                all_metrics |= all_metrics_at_k(ranked_rel, k, n_relevant)
                
            # Add the instance metrics into a list for the current evaluating model
            instances_metrics.append(all_metrics)
            
        # Average over all evaluated queries
        metrics_df = pd.DataFrame(instances_metrics)

        print("Model evaluation: \n")
        # print(f"  ✅  Done in {avg['encoding_time']}s")
        model_metrics = {}
        for k in K_VALUES:
            model_metrics[f"nDCG@{k}"] = metrics_df[f"nDCG@{k}"].mean()
            model_metrics[f"MAP@{k}"] = metrics_df[f"MAP@{k}"].mean(),
            model_metrics[f"MRR@{k}"] = metrics_df[f"MRR@{k}"].mean(),
            model_metrics[f"Recall@{k}"] = metrics_df[f"Recall@{k}"].mean(),
            model_metrics[f"Precision@{k}"] = metrics_df[f"Precision@{k}"].mean(),
            
        for key, value in model_metrics.items():
            print(f"*** {key}: ", value, " *** \n")

    except Exception as e:
        print(f"  ❌  Error: {e}")
        import traceback; traceback.print_exc()

print("\n🎉 All models evaluated.")

pyvi loaded — Vietnamese word segmentation available.

  AITeamVN  (AITeamVN/Vietnamese_Embedding)


Queries processed: 100%|██████████| 29746/29746 [16:00<00:00, 30.96it/s]


Model evaluation: 

*** nDCG@1:  0.5721777718012506  *** 

*** MAP@1:  (np.float64(0.5721777718012506),)  *** 

*** MRR@1:  (np.float64(0.5721777718012506),)  *** 

*** Recall@1:  (np.float64(0.5438423093301059),)  *** 

*** Precision@1:  (np.float64(0.5721777718012506),)  *** 

*** nDCG@5:  0.6864799152539148  *** 

*** MAP@5:  (np.float64(0.6458251623000665),)  *** 

*** MRR@5:  (np.float64(0.6659982294538201),)  *** 

*** Recall@5:  (np.float64(0.7870615657007104),)  *** 

*** Precision@5:  (np.float64(0.16971693673098903),)  *** 

*** nDCG@10:  0.7078508129548164  *** 

*** MAP@10:  (np.float64(0.6553060617816383),)  *** 

*** MRR@10:  (np.float64(0.6743075632951582),)  *** 

*** Recall@10:  (np.float64(0.8500521078464331),)  *** 

*** Precision@10:  (np.float64(0.09230148591407247),)  *** 


🎉 All models evaluated.


## 7. Results Summary

In [ ]:
metric_cols = [f"{m}@{k}"
               for m in ["nDCG", "MAP", "Recall", "MRR", "Precision"]
               for k in K_VALUES]

results_df = (
    pd.DataFrame(all_results)[["Model"] + metric_cols ]
    .sort_values("nDCG@10", ascending=False)
    .reset_index(drop=True)
)
results_df.index += 1
results_df.index.name = "Rank"

print("=" * 70)
print("  YuITC/Vietnamese-Legal-Documents — Retrieval Evaluation")
print("=" * 70)
results_df

In [ ]:
# nDCG@k view
ndcg_df = results_df[["Model"] + [f"nDCG@{k}" for k in K_VALUES]]
print("nDCG@k (higher is better):")
ndcg_df

## 8. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = ["#4C9BE8", "#F4845F", "#6BCB77", "#FFD166", "#C77DFF"]
models  = results_df["Model"].tolist()

In [ ]:
# ── Plot 1: Multi-metric bar chart at k=10 ────────────────────────
focus = ["nDCG@10", "MAP@10", "Recall@10", "MRR@10", "Precision@10"]

fig, ax = plt.subplots(figsize=(13, 5))
x, width = np.arange(len(models)), 0.15

for i, metric in enumerate(focus):
    bars = ax.bar(x + i * width, results_df[metric], width,
                  label=metric, color=PALETTE[i])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{bar.get_height():.3f}",
                ha="center", va="bottom", fontsize=7)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(models, rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.set_title("Vietnamese Legal Document Retrieval @10\nYuITC/Vietnamese-Legal-Documents")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("retrieval_at10.png", dpi=150)
plt.show()

In [ ]:
# ── Plot 2: nDCG@k line chart across cutoffs ──────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

for i, (_, row) in enumerate(results_df.iterrows()):
    ax.plot(K_VALUES, [row[f"nDCG@{k}"] for k in K_VALUES],
            marker="o", label=row["Model"], color=PALETTE[i % len(PALETTE)])

ax.set_xlabel("k")
ax.set_ylabel("nDCG@k")
ax.set_title("nDCG@k Across Cutoffs — Vietnamese Legal Retrieval")
ax.set_xticks(K_VALUES)
ax.legend()
plt.tight_layout()
plt.savefig("ndcg_vs_k.png", dpi=150)
plt.show()

In [ ]:
# ── Plot 3: Heatmap ────────────────────────────────────────────────
heatmap_cols = [f"{m}@{k}"
                for m in ["nDCG", "MAP", "Recall", "MRR"]
                for k in [1, 5, 10]]
heat_data = results_df.set_index("Model")[heatmap_cols].astype(float)

fig, ax = plt.subplots(figsize=(13, len(models) * 0.9 + 1.5))
sns.heatmap(heat_data, annot=True, fmt=".3f", cmap="YlGnBu",
            linewidths=0.4, ax=ax, cbar_kws={"label": "Score"})
ax.set_title("Retrieval Performance Heatmap — Vietnamese Legal Documents")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("retrieval_heatmap.png", dpi=150)
plt.show()

## 9. Error Analysis

In [ ]:
best_label = results_df.iloc[0]["Model"]
pq_df = per_query_all[best_label].copy()
print(f"Detailed error analysis for best model: {best_label}")
print(f"Queries evaluated: {len(pq_df):,}")

In [ ]:
# ── 9a: Rank distribution histogram ──────────────────────────────
rank_num = pq_df["rank_of_first_relevant"].apply(
    lambda x: 11 if x == ">10" else int(x)
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(rank_num, bins=range(1, 13), edgecolor="white",
        color="#4C9BE8", align="left", rwidth=0.85)
ax.set_xticks(range(1, 12))
ax.set_xticklabels([str(i) for i in range(1, 11)] + [">10"])
ax.set_xlabel("Rank of First Relevant Passage")
ax.set_ylabel("Number of Queries")
ax.set_title(f"Rank Distribution of First Hit — {best_label}")

hit1  = (rank_num == 1).sum()
hit10 = (rank_num <= 10).sum()
ax.text(0.97, 0.93,
        f"Hit@1  = {hit1}/{len(pq_df)} ({100*hit1/len(pq_df):.1f}%)\n"
        f"Hit@10 = {hit10}/{len(pq_df)} ({100*hit10/len(pq_df):.1f}%)",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round", fc="white", ec="gray"))
plt.tight_layout()
plt.savefig("rank_distribution.png", dpi=150)
plt.show()

In [ ]:
# ── 9b: nDCG@10 distribution per query ──────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(pq_df["nDCG@10"], bins=20, color="#6BCB77", edgecolor="white")
ax.axvline(pq_df["nDCG@10"].mean(), color="#F4845F", linestyle="--",
           label=f"Mean nDCG@10 = {pq_df['nDCG@10'].mean():.3f}")
ax.set_xlabel("nDCG@10")
ax.set_ylabel("Queries")
ax.set_title(f"nDCG@10 Distribution — {best_label}")
ax.legend()
plt.tight_layout()
plt.savefig("ndcg_distribution.png", dpi=150)
plt.show()

In [ ]:
# ── 9c: Worst queries ─────────────────────────────────────────────
pd.set_option("display.max_colwidth", 120)
failed = pq_df[pq_df["rank_of_first_relevant"] == ">10"].sort_values("nDCG@10")
print(f"Queries where relevant passage was NOT in top 10: "
      f"{len(failed)}/{len(pq_df)} ({100*len(failed)/len(pq_df):.1f}%)")
failed[["query", "relevant_passage_snippet", "top1_passage"]].head(10)

In [ ]:
# ── 9d: nDCG@10 vs number of relevant documents ──────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(pq_df["n_relevant"], pq_df["nDCG@10"],
           alpha=0.35, s=18, color="#4C9BE8")
ax.set_xlabel("Number of Relevant Passages per Query")
ax.set_ylabel("nDCG@10")
ax.set_title(f"Retrieval Difficulty vs. Relevant Docs — {best_label}")
plt.tight_layout()
plt.savefig("difficulty_scatter.png", dpi=150)
plt.show()

## 10. Hit Rate Comparison Across Models

In [ ]:
hit_rows = []
for label, pq in per_query_all.items():
    rn = pq["rank_of_first_relevant"].apply(
        lambda x: 11 if x == ">10" else int(x)
    )
    row = {"Model": label}
    for k in K_VALUES:
        row[f"Hit@{k}"] = round((rn <= k).mean(), 4)
    hit_rows.append(row)

hit_df = (
    pd.DataFrame(hit_rows)
    .sort_values("Hit@10", ascending=False)
    .reset_index(drop=True)
)
hit_df.index += 1
hit_df.index.name = "Rank"
print("=== Hit Rate (fraction of queries where first relevant passage is in top-k) ===")
hit_df

In [ ]:
hit_ks = [f"Hit@{k}" for k in K_VALUES]

fig, ax = plt.subplots(figsize=(11, 5))
x, width = np.arange(len(hit_df)), 0.18

for i, col in enumerate(hit_ks):
    bars = ax.bar(x + i * width, hit_df[col], width,
                  label=col, color=PALETTE[i % len(PALETTE)])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{bar.get_height():.2f}",
                ha="center", va="bottom", fontsize=7)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(hit_df["Model"], rotation=15, ha="right")
ax.set_ylabel("Hit Rate")
ax.set_ylim(0, 1.15)
ax.set_title("Hit Rate @ k — Vietnamese Legal Document Retrieval")
ax.legend()
plt.tight_layout()
plt.savefig("hit_rate.png", dpi=150)
plt.show()

## 11. Save Results

In [ ]:
results_df.to_csv("retrieval_results.csv")
hit_df.to_csv("hit_rate_results.csv")

for label, pq in per_query_all.items():
    safe = label.replace(" ", "_").replace("/","_").replace("(","").replace(")","")
    pq.to_csv(f"per_query_{safe}.csv", index=False)

print("Saved:")
print("  retrieval_results.csv   — aggregate metrics per model")
print("  hit_rate_results.csv    — Hit@k per model")
print("  per_query_<model>.csv   — per-query breakdown for each model")

---
## Quick Reference

| Setting | Where | Default |
|---|---|---|
| Number of queries to evaluate | Section 3 — `MAX_QUERIES` | `500` (set to `None` for all) |
| k cutoffs for metrics | Section 6 — `K_VALUES` | `[1, 5, 10, 20]` |
| Encoding batch size | Section 6 — `BATCH_SIZE` | `64` |
| Add / remove models | Section 4 — `MODELS` list | 5 models |

**Model guidance for Vietnamese legal retrieval:**
- `AITeamVN/Vietnamese_Embedding` — typically best for retrieval tasks; trained on Vietnamese triplets.
- `dangvantuan/vietnamese-document-embedding` — best if legal passages are long (supports 8096 tokens).
- `BAAI/bge-m3` — strong multilingual baseline; good if you later want cross-lingual retrieval.
- Inspect the `per_query_*.csv` files to understand failure patterns before choosing a model for production.